In [14]:
import json

with open("../data/ocr_pages.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

print("Number of pages:", len(pages))

Number of pages: 32


In [15]:
page = pages[2]

print("Page:", page["page"])
print()

for line in page["lines"]:
    print(line)

Page: 3

لا عمل إلا بنية
بسم الله الرحمن الرحيم
لا عمل إا بنية
عن أمير المؤمنين آبي حفص عمر بن
الخطاب رضي الله تعالى عنه قال ٠ سمعت رسول
الله صلى الله عليه وسلم يقول ٠ ) إنًمًا الأعمال بالنًيًات وإنًما لكلد
امرئ ما نوًى فمن كانت هجرته إلى اللًه ورسوله
فهجرته إلى اللًه ورسوله ومن كانت هجرته لدنيًا
يصيبها أو امرأة ينكحهًا ؛ فهجرته إل مًا هاجر إليه (
رواه إماما المحدثين أبو عبد اللًه محمد بن إسماعيل
ابن إبراهيم بن المغيرقة بن بردزبه البخاري م وأبو الحسين
مسلم بن الحجًاج بن مسلم القشيريً النًيسابوريً في
صحيحيهما اللذين هما أصح الكتب المصنًفة (١)
(١ أخرجه البخاري في بدء الوحي )ا( ومسلم في الإمارة )١٥٥(
قوله ٠ ل النيات ( أي القصد وعزم القلب على الفعل


In [16]:
import json
import os

from openai import OpenAI
from dotenv import load_dotenv

In [17]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY is not set.")

print("API key loaded successfully.")

API key loaded successfully.


In [18]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)
MODEL_NAME = "inclusionai/ling-3.0-flash-vl:free"

In [19]:
SYSTEM_PROMPT = """
You are an OCR correction assistant for a classical Arabic Islamic book
(Al-Arba'een Al-Nawawiyyah — a collection of 40 hadiths with commentary).

Your ONLY task is to fix OCR recognition errors. You are not an editor,
not a scholar, and not allowed to "improve" or "correct" the religious
text based on what you remember from other sources.

CONTEXT — the input line may be one of:
- A hadith number marker (e.g. "١ –", "٢ –") — keep exactly as is.
- A section/chapter title (originally inside a decorative box).
- The matn (body) of a hadith — narrator chain + Prophetic wording.
- A footnote: explanation of a word (غريب الحديث) or a reference/takhrij
  (e.g. "[رواه البخاري ومسلم]", numbers in parentheses like "(١)").
Do not merge or reorder these; treat the input as a single line/segment
and fix only what's inside it.

STRICT RULES:
1. Preserve the original text and meaning exactly.
2. Do NOT summarize, rewrite, or paraphrase.
3. Do NOT add or remove information, words, or phrases.
4. Do NOT change the order of words or sentences.
5. Do NOT "correct" wording to match a version of the hadith you recall
   from memory. If the input differs from what you remember but reads
   as a plausible original sentence, LEAVE IT UNCHANGED — this may be
   a real variant wording, not an OCR error.
6. Preserve narrator names, chains of narration (isnad), book titles,
   hadith references, and all religious terminology exactly as spelled,
   unless clearly a letter-level OCR glitch (see examples below).
7. Diacritics (tashkeel): only fix a diacritic mark if it is obviously
   broken/garbled (e.g. a stray unrecognizable symbol in place of a
   harakah). Do NOT add diacritics that are missing — leaving a word
   undiacritized is always safer than guessing a diacritic.
8. Fix only clear, low-level OCR artifacts:
   - misrecognized/confused Arabic letters (e.g. ه/ة, ي/ى, ب/ت/ث)
   - missing or duplicated letters within a word
   - broken/merged word spacing
   - garbled punctuation
   - corrupted decorative glyphs — see examples below
9. If you are not confident something is an OCR error, leave it
   UNCHANGED. Never guess.
10. Output ONLY the corrected Arabic text — no explanations, no
    markdown, no quotation marks, no comments.

EXAMPLES OF KNOWN OCR GLITCHES IN THIS BOOK (apply the same pattern to
similar garbled tokens near the Prophet's name or at the top of a page,
even if the exact garbled letters differ):
- Garbled token appearing right after mentions of the Prophet (from a
  misread ﷺ ligature), e.g. "عطقه", "عقطته", "علقاهم" → "صلى الله عليه وسلم"
- A large garbled token at the very top of a hadith/page (from a
  misread Basmalah ornament), e.g. "س إلقوالثفز قليي" → "بسم الله الرحمن الرحيم"
- If you see a short, meaningless token in a position where a religious
  formula (صلى الله عليه وسلم / رضي الله عنه / تعالى) would normally
  appear, and it does not form a real Arabic word, treat it as a
  corrupted formula and restore the standard phrasing that fits the
  grammatical context — but only when no real alternative reading of
  the token as an actual word is plausible.
"""

In [23]:
def correct_page_with_llm(page_lines):
    page_text = "\n".join(page_lines)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": page_text
            }
        ],
        temperature=0
    )

    message = response.choices[0].message

    if message.content is None:
        print("LLM returned no text.")
        print("Response:")
        print(response)
        return None

    return message.content.strip()

In [21]:
test_page = pages[2]

corrected_text = correct_page_with_llm(test_page["lines"])

print(corrected_text)

لا عمل إلا بنية
بسم الله الرحمن الرحيم
لا عمل إلا بنية
عن أمير المؤمنين أبى حفص عمر بن
الخطاب رضي الله تعالى عنه قال ٠ سمعت رسول
الله صلى الله عليه وسلم يقول ٠ ( إنَّمَا الأعمال بالنِّيَّاتِ وإنَّمَا لكلِّ
امرئ ما نَوَى فمن كانت هجرته إلى اللَّهِ ورسوله
فهجرته إلى اللَّهِ ورسوله ومن كانت هجرته لدنيًا
يصيبها أو امرأة ينكحهًا ؛ فهجرته إلَّمَا هاجر إليه )
رواه إماما المحدثين أبو عبد اللَّهِ محمد بن إسماعيل
ابن إبراهيم بن المغيرقة بن بردزة البخاري م وأبو الحسين
مسلم بن الحَجَّاجِ بن مسلم القشيرِيِّ النَّيسَابُورِيِّ في
صحيحيهما اللذين هما أصح الكتب المُصَفَّفَةِ (١)
(١ أخرجه البخاري في بدء الوحي (ا) ومسلم في الإمارة (١٥٥)
قوله ٠ للنيات ( أي القصد وعزم القلب على الفعل


In [25]:
cleaned_pages = []

for page in pages[9:]:
    print(f"Processing page {page['page']}...")

    corrected_text = correct_page_with_llm(page["lines"])

    if corrected_text is None:
        print(f"Skipping page {page['page']} because LLM returned no text.")
        break

    cleaned_pages.append({
        "page": page["page"],
        "text": corrected_text
    })

print("Finished.")

Processing page 10...
Processing page 11...
Processing page 12...
Processing page 13...
Processing page 14...
Processing page 15...
Processing page 16...
Processing page 17...
Processing page 18...
Processing page 19...
Processing page 20...
Processing page 21...
Processing page 22...
Processing page 23...
Processing page 24...
Processing page 25...
Processing page 26...
Processing page 27...
Processing page 28...
Processing page 29...
Processing page 30...
Processing page 31...
Processing page 32...
Finished.


In [27]:
output_path = "../data/cleaned_pages.json"

# اقرأ البيانات الموجودة لو الملف موجود
if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        existing_pages = json.load(f)
else:
    existing_pages = []

# الصفحات الموجودة بالفعل
existing_page_numbers = {
    page["page"] for page in existing_pages
}

# أضف فقط الصفحات الجديدة
for page in cleaned_pages:
    if page["page"] not in existing_page_numbers:
        existing_pages.append(page)

# احفظ كل البيانات مرة أخرى
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        existing_pages,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Total pages saved: {len(existing_pages)}")
print(f"Saved to: {output_path}")

Total pages saved: 32
Saved to: ../data/cleaned_pages.json
